In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('phishing_site_urls.csv')

# Display the first few rows
print(df.head())

# Check for missing values
print(df.isnull().sum())

# Examine the distribution of labels
print(df['Label'].value_counts())


In [ ]:
# === Extract domain from URL ===
def clean_domain_from_url(url):
    try:
        url = url.strip().lower()
        if not url.startswith("http"):
            url = "http://" + url
        return urlparse(url).netloc.split(':')[0]
    except:
        return None



In [15]:
# === Extract basic domain features ===
def extract_domain_properties(domain):
    if not isinstance(domain, str):
        return pd.Series([None, None, None])
    domain_length = len(domain)
    has_digits = any(char.isdigit() for char in domain)
    has_hyphen = '-' in domain
    return pd.Series([domain_length, int(has_digits), int(has_hyphen)])

df[['domain_length', 'has_digits', 'has_hyphen']] = df['domain'].apply(extract_domain_properties)


In [13]:
print(df.head())

                                                 URL Label  \
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...   bad   
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...   bad   
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....   bad   
3  mail.printakid.com/www.online.americanexpress....   bad   
4  thewhiskeydregs.com/wp-content/themes/widescre...   bad   

                domain  
0            nobell.it  
1      www.dghjdgf.com  
2     serviciosbys.com  
3   mail.printakid.com  
4  thewhiskeydregs.com  


In [16]:
# === Get IP Address ===
def get_ip(domain):
    try:
        socket.setdefaulttimeout(3)
        ip = socket.gethostbyname(domain)
        return ip
    except:
        return None

with ThreadPoolExecutor(max_workers=100) as executor:
    df['ip'] = list(executor.map(get_ip, df['domain']))

In [17]:
# === Save intermediate result ===
df.to_csv('url_with_ip_and_domain_features.csv', index=False)

In [18]:
# === WHOIS Lookup ===
def get_whois_info(domain):
    try:
        w = whois.whois(domain)
        return {
            'domain': domain,
            'creation_date': w.creation_date if isinstance(w.creation_date, datetime) else None,
            'registrar': w.registrar,
            'country': w.country,
            'city': getattr(w, 'city', None)
        }
    except:
        return {
            'domain': domain,
            'creation_date': None,
            'registrar': None,
            'country': None,
            'city': None
        }


In [ ]:
import pandas as pd
import whois
from datetime import datetime
from concurrent.futures import as_completed, ThreadPoolExecutor
import time
import os

# ----------------s----------
# CONFIGURATION
# --------------------------
batch_size = 5000
batch_start = 0     # Change this if continuing from a later batch
max_workers = 5
source_file = "url_with_ip_and_domain_features.csv"
output_file = "whois_master.csv"  # Final merged file
log_file = "whois_progress.log"   # Tracks progress


# Load data
df = pd.read_csv(source_file)
df = df[df['ip'].notnull()]
df['domain'] = df['domain'].apply(clean_domain)
domains = df['domain'].dropna().unique().tolist()
print(df['domain'].head(10).tolist())


In [ ]:
# Load previous progress
if os.path.exists(output_file):
    done_df = pd.read_csv(output_file)
    done_domains = set(done_df['domain'].tolist())
    print(f"✅ Already processed: {len(done_domains)} domains")
else:
    done_df = pd.DataFrame()
    done_domains = set()

remaining_domains = [d for d in domains if d not in done_domains]
print(f"🕒 Domains remaining: {len(remaining_domains)}")

# --------------------------
# Process in batches
# --------------------------
for batch_start in range(0, len(remaining_domains), batch_size):
    batch_end = min(batch_start + batch_size, len(remaining_domains))
    current_batch = remaining_domains[batch_start:batch_end]
    print(f"\n🚀 Processing batch {batch_start}-{batch_end} ...")

    start_time = time.time()
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_domain = {executor.submit(get_whois_info, domain): domain for domain in current_batch}
        for future in as_completed(future_to_domain):
            result = future.result()
            results.append(result)

    batch_df = pd.DataFrame(results)
    
    # Append to master file
    if os.path.exists(output_file):
        batch_df.to_csv(output_file, mode='a', header=False, index=False)
    else:
        batch_df.to_csv(output_file, index=False)

    print(f"✅ Batch {batch_start}-{batch_end} done in {round((time.time() - start_time)/60, 2)} minutes")
    
    # Save progress
    with open(log_file, 'a') as f:
        f.write(f"Batch {batch_start}-{batch_end} completed\n")

2025-06-05 18:09:31,802 - whois.whois - ERROR - Error trying to connect to socket: closing socket - timed out
ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out
2025-06-05 18:09:59,734 - whois.whois - ERROR - Error trying to connect to socket: closing socket - timed out
ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out
2025-06-05 18:10:06,534 - whois.whois - ERROR - Error trying to connect to socket: closing socket - timed out
ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out
2025-06-05 18:10:07,974 - whois.whois - ERROR - Error trying to connect to socket: closing socket - timed out
ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out
2025-06-05 18:10:23,272 - whois.whois - ERROR - Error trying to connect to socket: closing socket - timed out
ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out
2025-06-05 18:10:26,143 - whois.whois - ERROR - Er

In [ ]:
df_main = pd.read_csv("urls_with_ip_and_domain_features.csv")
whois_df = pd.read_csv("whois_master.csv")

final_df = pd.merge(df_main, whois_df, on='domain', how='left')
final_df.to_csv("urls_full_with_ip_domain_and_whois.csv", index=False)